# Aggregation Method and Training Data Ablations

In the following we demonstrate how to reproduce the aggregation method ablations (using a KDE instead of the proposed max-quantile strategy to combine the individual metrics). Additionally, both approaches are evaluated using either the training or validation data for the fitting.

In [ ]:
import pandas as pd

from sitn.aggregators import KDE, MaxQuantile
from sitn.metrics import bootstrap_auroc
from sitn.utils import construct_results_path

In [ ]:
# Configurations
# We assume the model has already be trained and evaluated with
# these configurations (follow the cross-dataset OOD detection
# notebook to see how).

train_cfg = {"dataset_name": "cifar10"}

eval_cfg_train = {"config": train_cfg, "split_pick": "train"}
eval_cfg_val = {"config": train_cfg, "split_pick": "val"}
eval_cfg_test = {"config": train_cfg, "split_pick": "test"}

eval_cfg_svhn = {"config": train_cfg, "eval_dataset_name": "svhn", "split_pick": "test"}
eval_cfg_celeba = {"config": train_cfg, "eval_dataset_name": "celeba", "split_pick": "test"}

In [ ]:
# Load train and val ID predictions
id_train_preds = pd.read_csv(construct_results_path(**eval_cfg_train, result_type="predictions"))
id_val_preds = pd.read_csv(construct_results_path(**eval_cfg_val, result_type="predictions"))

# Fit SITN (original version)
sitn = MaxQuantile({"anderson_darling_statistic": True, "ps_cv": True})
sitn.fit(id_val_preds)

# Fit SITN on train data
sitn_train = MaxQuantile({"anderson_darling_statistic": True, "ps_cv": True})
sitn_train.fit(id_train_preds)

# Fit SITN using KDE instead of max quantile on train data
sitn_kde_train = KDE(features=["anderson_darling_statistic", "ps_cv"])
sitn_kde_train.fit(id_train_preds)

# Fit SITN using KDE instead of max quantile on val data
sitn_kde_val = KDE(features=["anderson_darling_statistic", "ps_cv"])
sitn_kde_val.fit(id_val_preds)

In [ ]:
metrics = {
    "sitn": {"label": "SITN", "higher_is_ood": True},
    "sitn_train": {"label": "SITN (train)", "higher_is_ood": True},
    "sitn_kde_val": {"label": "SITN-KDE (val)", "higher_is_ood": False},
    "sitn_kde_train": {"label": "SITN-KDE (train)", "higher_is_ood": False},
}

# Load ID test predictions
id_preds = pd.read_csv(construct_results_path(**eval_cfg_test, result_type="predictions"))
id_preds["train_dataset"] = eval_cfg_test["config"]["dataset_name"]
id_preds["eval_dataset"] = eval_cfg_test["config"]["dataset_name"]

results = []
for eval_cfg_ood in [eval_cfg_svhn, eval_cfg_celeba]:
    # Load OOD test predictions
    ood_preds = pd.read_csv(construct_results_path(**eval_cfg_ood, result_type="predictions"))
    ood_preds["train_dataset"] = eval_cfg_ood["config"]["dataset_name"]
    ood_preds["eval_dataset"] = eval_cfg_ood["eval_dataset_name"]

    # Combine ID and OOD predictions
    preds = pd.concat([id_preds.copy(), ood_preds], ignore_index=True)

    # Add original and ablation versions of SITN score
    preds["sitn"] = sitn.score(preds)
    preds["sitn_train"] = sitn_train.score(preds)
    preds["sitn_kde_val"] = sitn_kde_val.score(preds)
    preds["sitn_kde_train"] = sitn_kde_train.score(preds)

    # Compute AUROC with bootstrapped CIs for each method
    y_true = (preds["eval_dataset"] != preds["train_dataset"]).astype(int)
    for col, meta in metrics.items():
        scores = preds[col].copy()
        if not meta["higher_is_ood"]:
            scores = -scores

        auroc, ci_lo, ci_hi = bootstrap_auroc(y_true, scores)
        results.append(
            {
                "ood_dataset": eval_cfg_ood["eval_dataset_name"],
                "metric": meta["label"],
                "AUROC": auroc,
                "CI_lo": ci_lo,
                "CI_hi": ci_hi,
            }
        )

results = pd.DataFrame(results).set_index(["ood_dataset", "metric"])
results

AUROC     CI_lo     CI_hi
ood_dataset metric                                        
svhn        SITN              0.946121  0.943365  0.948894
            SITN (train)      0.948208  0.945529  0.950910
            SITN-KDE (val)    0.914796  0.911075  0.918448
            SITN-KDE (train)  0.912929  0.909182  0.916575
celeba      SITN              0.671056  0.664537  0.677723
            SITN (train)      0.672493  0.666009  0.679167
            SITN-KDE (val)    0.614529  0.607811  0.621363
            SITN-KDE (train)  0.616936  0.610292  0.623773